### Testing cubes/construct package

This is a notebook to perform a test to determine if the construct package can produce a reliable idf file

In [1]:
from cubes.construct import building
from cubes.construct import sample
from cubes.construct.extractor import Extractor
from cubes.construct import constants
from cubes.construct.building import Building
from cubes.package.simple_simulation import prepare_simulation
from cubes.package import envconfig
from geomeppy import IDF
import pandas as pd

import os
from pathlib import Path

In [2]:
def sample_idf():
    geometry_data, systems_data = sample.sample_database(
        constants.raw_geometry_data, constants.clean_system_data
    )
    extracted = Extractor(geometry_data, systems_data)
    building_config = extracted.create_building_config_object()
    build = building.Building(building_config)
    build.build()
    idf = build.get_idf()

    return idf

def test_specific_idf(building_code):
    geometry_data = constants.raw_geometry_data[constants.raw_geometry_data['REFERENCE BUILDING CODE'] == building_code]
    systems_data =constants.clean_system_data[constants.clean_system_data['Building typology'] == building_code]

    geometry_data = geometry_data.reset_index(drop = True)

    #geometry_data, systems_data = sample.sample_database(
    #    constants.raw_geometry_data, constants.clean_system_data
    #)
    extracted = Extractor(geometry_data, systems_data)

    BC = extracted.create_building_config_object()
    EC = envconfig.EnvConfig(observe_zone_temperature=True,observe_electricity_demand=True,observe_outside_temperature=True)    
    building=Building(BC)
    building.build()
    idf = building.get_idf()

    cwd_path = os.getcwd()
    env_data_path = os.path.join(cwd_path, "input_case_1")
    output_directory_path = os.path.join(env_data_path, "output")

    Path(env_data_path).mkdir(parents=True, exist_ok=True)
    Path(output_directory_path).mkdir(parents=True, exist_ok=True)

    idf,weather_file=prepare_simulation(idf,BC,EC)
    
    idf.save(filename=env_data_path + "/test1.idf")
    print(weather_file)
    idf.run(
        expandobjects=True,
        readvars=True,
        weather=weather_file,
        output_directory=output_directory_path,
)

def test_idf():

    idf1 = sample_idf()
    cwd_path = os.getcwd()
    env_data_path = os.path.join(cwd_path, "input_case_1")
    output_directory_path = os.path.join(env_data_path, "output")

    Path(env_data_path).mkdir(parents=True, exist_ok=True)
    Path(output_directory_path).mkdir(parents=True, exist_ok=True)

    idf1.save(filename=env_data_path + "/test1.idf")

    IDF.setiddname("/usr/local/EnergyPlus-9-5-0/Energy+.idd")
    idf = IDF("/workspaces/CUBES/exp/jack/construct-tests/input_case_1/test1.idf")
    idf.epw = "/workspaces/CUBES/src/cubes/data/weather/cambridge_lat=52.25_lng=0.25_period=2021.epw"

    idf.run(
        expandobjects=True,
        output_directory=output_directory_path,
    )

def test_walls():
    geometry_data, systems_data = sample.sample_database(
        constants.filtered_geometry_data, constants.clean_system_data
    )
    building_config = Extractor(geometry_data, systems_data)
    walls = building_config.calc_wall_length()
    
    return walls, geometry_data
    
def get_minimal_idf():
    geometry_data, systems_data = sample.sample_database(
        constants.filtered_geometry_data, constants.clean_system_data
    )
    building_config = Extractor(geometry_data, systems_data)
    build = building.Building(building_config)
    idf = build.get_idf()
    return idf

def get_specific_archetype(building_code):
    geometry_data = constants.raw_geometry_data[constants.raw_geometry_data['REFERENCE BUILDING CODE'] == building_code]
    systems_data =constants.clean_system_data[constants.clean_system_data['Building typology'] == building_code]

    geometry_data = geometry_data.reset_index(drop = True)
    return geometry_data
    
def test_specific_archetype(building_code):
    geometry_data = constants.raw_geometry_data[constants.raw_geometry_data['REFERENCE BUILDING CODE'] == building_code]
    systems_data =constants.clean_system_data[constants.clean_system_data['Building typology'] == building_code]

    geometry_data = geometry_data.reset_index(drop = True)
    systems_data = systems_data.reset_index(drop = True)

    extracted = Extractor(geometry_data, systems_data)
    building_config = extracted.create_building_config_object()
    build = building.Building(building_config)
    build.build()
    idf1 = build.get_idf()
    
    cwd_path = os.getcwd()
    env_data_path = os.path.join(cwd_path, "input_case_1")
    output_directory_path = os.path.join(env_data_path, "output")

    Path(env_data_path).mkdir(parents=True, exist_ok=True)
    Path(output_directory_path).mkdir(parents=True, exist_ok=True)

    idf1.save(filename=env_data_path + "/test1.idf")

    IDF.setiddname("/usr/local/EnergyPlus-9-5-0/Energy+.idd")
    idf = IDF("/workspaces/CUBES/exp/jack/construct-tests/input_case_1/test1.idf")      
    idf.epw = "/workspaces/CUBES/src/cubes/data/weather/cambridge_lat=52.25_lng=0.25_period=2021.epw" 
    idf.run(
        expandobjects=True,
        output_directory=output_directory_path,     )    
    
    return idf1

    

In [14]:
dtls = idf.model.dtls
len(dtls)

821

In [23]:
objname = dtls[213]
objname
idf.idfobjects[objname]

TypeError: int() argument must be a string, a bytes-like object or a number, not 'tuple'

In [24]:
objname

'ELECTRICEQUIPMENT'

In [16]:
astr = ""
dtls = idf.model.dtls
for index, objname in enumerate(dtls):
    print(index)
    for obj in idf.idfobjects[objname]:
        astr = astr + obj.__repr__()
        

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212


TypeError: int() argument must be a string, a bytes-like object or a number, not 'tuple'

In [8]:
test_specific_archetype("FR-SFH-2000-2005-00")

13    ZoneHVAC:LowTemperatureRadiant:Electric 
dtype: object Electricity


AttributeError: 'IDF' object has no attribute 'epw'

In [ ]:
walls = {'Country': [], 'Code': [], 'wall_lengths': []}
for m in range(0,100):
    wall, gm = test_walls()
    walls['Country'].append(gm['REFERENCE BUILDING COUNTRY CODE'].values[0])
    walls['Code'].append(gm['REFERENCE BUILDING CODE'].values[0])
    walls['wall_lengths'].append(math.isnan(wall[0]))

df = pd.DataFrame.from_dict(walls)
df[df['wall_lengths'] == True]